# Assistiv Systems — CBI Data Fetcher
**Carer Burden Index · Layer 2 companion · Kent & Medway · v1.0**

Produces `kent-cbi-data.json` alongside the FEP JSON in the `assistivagents` repo.

### Signals:
| # | Signal | Source | Geography | Weight |
|---|--------|--------|-----------|--------|
| 1 | 20+ hr unpaid carers % | ONS Census 2021 TS039 | **District** | 30% |
| 2 | Carer social isolation | Fingertips 90638 | County | 25% |
| 3 | Carer support info access | Fingertips 90279 | County | 15% |
| 4 | Carer assessment rate | Fingertips 93014 | County | 15% |
| 5 | Carer wellbeing score | Fingertips 90283 | County | 15% |

### Methodology notes:
- Signals 2–5 are at Kent County level — applied uniformly across all 13 districts
- Signal 1 (Census) provides the district-level differentiation
- DWP Carer's Allowance can be added as Signal 6 once Stat-Xplore download is complete
- Antidepressant/anxiolytic prescribing deliberately excluded — already in FEP model
- All signals inverted where appropriate: lower score = higher carer burden

### Before running:
- Add `GITHUB_TOKEN` to Colab Secrets (🔑 sidebar) with `public_repo` scope
- `Runtime → Run all` — takes approximately 2 minutes (no EPD streaming)

In [ ]:
import subprocess
subprocess.run(['pip','install','fingertips_py','requests','pandas','--quiet'], check=True)
print('Dependencies ready')

import requests, pandas as pd, fingertips_py as ftp
import json, base64
from datetime import datetime, timezone
from google.colab import userdata

GITHUB_REPO  = 'silegrand/assistivagents'
GITHUB_FILE  = 'kent-cbi-data.json'
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()
print(f'GitHub token: {"Found" if GITHUB_TOKEN else "MISSING"}')

KENT_COUNTY = 'E10000016'  # Kent County Council — finest geography for carer indicators
ENGLAND     = 'E92000001'
BASE_FT     = 'https://fingertips.phe.org.uk/api'

# District name map — ONS code → display name
DISTRICTS = {
    'E07000114': 'Thanet',
    'E07000112': 'Folkestone & Hythe',
    'E07000108': 'Dover',
    'E07000113': 'Swale',
    'E06000035': 'Medway',
    'E07000109': 'Gravesham',
    'E07000105': 'Ashford',
    'E07000106': 'Canterbury',
    'E07000107': 'Dartford',
    'E07000110': 'Maidstone',
    'E07000115': 'Tonbridge & Malling',
    'E07000111': 'Sevenoaks',
    'E07000116': 'Tunbridge Wells',
}
ALL_DISTRICTS = list(DISTRICTS.values())

# ── Signal weights — sum to 1.0 ───────────────────────────────────────────────
WEIGHTS = {
    'census_20plus':    0.30,  # ONS Census 2021 — district level, primary differentiator
    'social_isolation': 0.25,  # Fingertips 90638 — strongest single burnout predictor
    'support_access':   0.15,  # Fingertips 90279 — support infrastructure
    'assessment_rate':  0.15,  # Fingertips 93014 — statutory system engagement
    'wellbeing':        0.15,  # Fingertips 90283 — direct carer wellbeing
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 0.001, f'Weights sum to {sum(WEIGHTS.values()):.4f}'
print(f'Weights check: {sum(WEIGHTS.values()):.4f} ✓')

# ── Census 2021 TS039 — pre-extracted district data ───────────────────────────
# Source: ONS Census 2021 TS039 LTLA bulk download (Nomis)
# England average 20+ hr carers: 4.5187% of population aged 5+
ENG_20HRS_PCT = 4.5187

CENSUS_DATA = {
    'Thanet':              {'total':133253,'hrs_9_less':4100, 'hrs_10_19':1772,'hrs_20_49':2899,'hrs_50_plus':5076,'high_hrs':7975, 'pct_20plus':5.9849},
    'Folkestone & Hythe':  {'total':104645,'hrs_9_less':3537, 'hrs_10_19':1446,'hrs_20_49':2168,'hrs_50_plus':3671,'high_hrs':5839, 'pct_20plus':5.5798},
    'Swale':               {'total':142781,'hrs_9_less':4339, 'hrs_10_19':1722,'hrs_20_49':2884,'hrs_50_plus':4984,'high_hrs':7868, 'pct_20plus':5.5105},
    'Dover':               {'total':110741,'hrs_9_less':3791, 'hrs_10_19':1574,'hrs_20_49':2344,'hrs_50_plus':3794,'high_hrs':6138, 'pct_20plus':5.5427},
    'Gravesham':           {'total':100299,'hrs_9_less':2933, 'hrs_10_19':1208,'hrs_20_49':1876,'hrs_50_plus':2969,'high_hrs':4845, 'pct_20plus':4.8306},
    'Medway':              {'total':262465,'hrs_9_less':7265, 'hrs_10_19':3041,'hrs_20_49':4932,'hrs_50_plus':7380,'high_hrs':12312,'pct_20plus':4.6909},
    'Ashford':             {'total':125137,'hrs_9_less':3963, 'hrs_10_19':1487,'hrs_20_49':2199,'hrs_50_plus':3501,'high_hrs':5700, 'pct_20plus':4.5550},
    'Canterbury':          {'total':150606,'hrs_9_less':5003, 'hrs_10_19':1740,'hrs_20_49':2478,'hrs_50_plus':4163,'high_hrs':6641, 'pct_20plus':4.4095},
    'Maidstone':           {'total':165392,'hrs_9_less':5484, 'hrs_10_19':1886,'hrs_20_49':2567,'hrs_50_plus':4245,'high_hrs':6812, 'pct_20plus':4.1187},
    'Tonbridge & Malling': {'total':124520,'hrs_9_less':4341, 'hrs_10_19':1384,'hrs_20_49':1947,'hrs_50_plus':3158,'high_hrs':5105, 'pct_20plus':4.0997},
    'Dartford':            {'total':108471,'hrs_9_less':3107, 'hrs_10_19':1155,'hrs_20_49':1700,'hrs_50_plus':2701,'high_hrs':4401, 'pct_20plus':4.0573},
    'Sevenoaks':           {'total':113792,'hrs_9_less':4234, 'hrs_10_19':1304,'hrs_20_49':1648,'hrs_50_plus':2797,'high_hrs':4445, 'pct_20plus':3.9062},
    'Tunbridge Wells':     {'total':109144,'hrs_9_less':3825, 'hrs_10_19':1127,'hrs_20_49':1410,'hrs_50_plus':2253,'high_hrs':3663, 'pct_20plus':3.3561},
}

print(f'Config loaded | {len(DISTRICTS)} districts | {len(WEIGHTS)} signals')
print(f'Census data: {len(CENSUS_DATA)} districts embedded (ONS Census 2021 TS039)')
print(f'England 20+ hr carer average: {ENG_20HRS_PCT}%')

## Cell 2 — Fetch Fingertips Carer Indicators
Fetches four signals at Kent County level using the Fingertips REST API directly.
England averages are fetched live — no hardcoded reference values.

All four signals are **inverted**: a lower value means worse outcome for Kent carers,
which normalises to a higher CBI contribution.

In [ ]:
def ft_fetch(ind_id, area_code):
    """Fetch latest value for indicator at area_code via Fingertips REST API."""
    try:
        r = requests.get(
            f'{BASE_FT}/latest_data/by_indicator_ids'
            f'?indicator_ids={ind_id}&area_codes={area_code}',
            timeout=15
        ).json()
        rows = sorted(r.get(area_code, []), key=lambda x: x.get('TimePeriodSortable', 0))
        if not rows: return None, None
        return rows[-1].get('Value'), rows[-1].get('TimePeriod', '?')
    except Exception as e:
        return None, str(e)

def ft_fetch_fpy(ind_id, area_code):
    """Fallback: fingertips_py with area code filter."""
    try:
        data = ftp.get_data_for_indicator_at_all_available_geographies(ind_id)
        if data is None: return None, 'None returned'
        rows = data[data['Area Code'] == area_code].sort_values('Time period')
        if len(rows) == 0: return None, f'no rows for {area_code}'
        return float(rows.tail(1)['Value'].values[0]), str(rows.tail(1)['Time period'].values[0])
    except Exception as e:
        return None, str(e)

FT_INDICATORS = {
    90638: {'key':'social_isolation', 'label':'Carer social isolation',      'invert':True},
    90279: {'key':'support_access',   'label':'Carer support info access',   'invert':True},
    93014: {'key':'assessment_rate',  'label':"Carer's Assessment rate",     'invert':False},
    90283: {'key':'wellbeing',        'label':'Carer wellbeing score',        'invert':False},
}

fingertips = {}
print('Fetching Fingertips carer indicators...')
print(f'{"ID":<8} {"Label":<32} {"Kent":>8}  {"England":>8}  {"Δ%":>8}  Period    Method')
print('-' * 85)

for ind_id, meta in FT_INDICATORS.items():
    # Fetch Kent
    kent_val, kent_period = ft_fetch(ind_id, KENT_COUNTY)
    method = 'REST'
    if kent_val is None:
        kent_val, kent_period = ft_fetch_fpy(ind_id, KENT_COUNTY)
        method = 'ftp_py'

    # Fetch England
    eng_val, _ = ft_fetch(ind_id, ENGLAND)
    if eng_val is None:
        eng_val, _ = ft_fetch_fpy(ind_id, ENGLAND)

    delta = f'{((kent_val-eng_val)/eng_val)*100:+.1f}%' if kent_val and eng_val else 'n/a'

    # For inverted signals: lower kent = worse
    if kent_val and eng_val:
        worse = (kent_val < eng_val) if meta['invert'] else (kent_val > eng_val)
        flag  = '⚠' if worse else '✓'
    else:
        flag = '?'

    print(f'{ind_id:<8} {meta["label"]:<32} '
          f'{str(round(kent_val,2)) if kent_val else "n/a":>8}  '
          f'{str(round(eng_val,2)) if eng_val else "n/a":>8}  '
          f'{delta:>8}  {str(kent_period):<10}  {flag} [{method}]')

    fingertips[meta['key']] = {
        'indicator_id': ind_id,
        'label':        meta['label'],
        'kent_val':     kent_val,
        'england_val':  eng_val,
        'period':       kent_period,
        'invert':       meta['invert'],
        'source':       f'NHS Fingertips indicator {ind_id} — Kent County (E10000016)',
        'geo_note':     'County level — applied uniformly across all 13 districts',
    }

fetched = sum(1 for v in fingertips.values() if v['kent_val'])
print(f'\n{fetched}/{len(FT_INDICATORS)} Fingertips indicators fetched')
print('Note: all at Kent County (E10000016) — finest available geography for these indicators')

## Cell 3 — Build District CBI Scores & Commit

**Normalisation approach:**
- Each signal normalised to 0–100 where 50 = England average
- Score > 50 = higher carer burden than England average
- Census signal: district-specific (real variation between districts)
- Fingertips signals: county-level value applied to all districts
  (county signals shift all districts equally — Census drives the within-county spread)

**Risk tiers:**
- Critical ≥ 70 · High ≥ 55 · Moderate ≥ 40 · Low < 40

**Dual-score output:**
A district's FEP × CBI quadrant is the primary commissioning signal.
High FEP + High CBI = both older adults and their carers at risk simultaneously.

In [ ]:
def norm(kent_val, eng_val, invert=False):
    """Normalise to 0-100 where 50 = England average.
    Invert=True: lower value = worse = higher score (used for social isolation etc.)"""
    if not kent_val or not eng_val: return 50.0
    ratio = kent_val / eng_val
    if invert:
        # Lower than England = higher burden — invert the ratio
        score = (2 - ratio) * 50
    else:
        score = ratio * 50
    return round(min(100, max(0, score)), 1)

def norm_census(district_pct, eng_pct):
    """Normalise 20+ hr carer % against England average."""
    if not district_pct or not eng_pct: return 50.0
    return round(min(100, max(0, (district_pct / eng_pct) * 50)), 1)

# ── County-level Fingertips normalised scores ─────────────────────────────────
# Same for all districts — county signals shift the baseline uniformly
ft_scores = {}
for key, v in fingertips.items():
    ft_scores[key] = norm(v['kent_val'], v['england_val'], invert=v['invert'])

print('County-level Fingertips normalised scores (applied to all districts):')
for key, score in ft_scores.items():
    label = fingertips[key]['label']
    w     = WEIGHTS[key]
    contribution = round(score * w, 2)
    print(f'  {label:<35} {score:>6.1f}  weight:{w:.0%}  contribution:{contribution:.1f}')
ft_county_contribution = sum(ft_scores.get(k, 50) * WEIGHTS[k]
                             for k in ['social_isolation','support_access','assessment_rate','wellbeing'])
print(f'\nTotal county-level contribution to CBI (before Census): {ft_county_contribution:.1f}')
print(f'Census (district-level) contributes remaining 30% — drives within-county spread')

# ── LAD codes ─────────────────────────────────────────────────────────────────
LAD_CODES = {
    'Thanet':'E07000114','Folkestone & Hythe':'E07000112','Dover':'E07000108',
    'Swale':'E07000113','Medway':'E06000035','Gravesham':'E07000109',
    'Ashford':'E07000105','Canterbury':'E07000106','Dartford':'E07000107',
    'Maidstone':'E07000110','Tonbridge & Malling':'E07000115',
    'Sevenoaks':'E07000111','Tunbridge Wells':'E07000116',
}

# ── Build district scores ──────────────────────────────────────────────────────
districts = []
print(f'\n{"District":<25} {"Census":>7}  {"FT":>7}  {"CBI":>6}  {"Tier":<10}  20+%  50+carers')
print('-' * 80)

for name in ALL_DISTRICTS:
    c = CENSUS_DATA[name]
    census_score = norm_census(c['pct_20plus'], ENG_20HRS_PCT)

    # Weighted CBI
    signal_scores = {
        'census_20plus':    census_score,
        'social_isolation': ft_scores.get('social_isolation', 50),
        'support_access':   ft_scores.get('support_access', 50),
        'assessment_rate':  ft_scores.get('assessment_rate', 50),
        'wellbeing':        ft_scores.get('wellbeing', 50),
    }
    cbi = round(sum(signal_scores[k] * WEIGHTS[k] for k in WEIGHTS))
    tier = ('critical' if cbi >= 70 else 'high' if cbi >= 55
            else 'moderate' if cbi >= 40 else 'low')

    ft_total = round(sum(ft_scores.get(k,50)*WEIGHTS[k]
                         for k in ['social_isolation','support_access','assessment_rate','wellbeing']), 1)

    print(f'{name:<25} {census_score:>7.1f}  {ft_total:>7.1f}  {cbi:>6}  {tier:<10}  '
          f'{c["pct_20plus"]:>4.2f}%  {c["hrs_50_plus"]:,}')

    districts.append({
        'name':           name,
        'lad_code':       LAD_CODES[name],
        'cbi':            cbi,
        'tier':           tier,
        'signal_scores':  signal_scores,
        'signal_weights': WEIGHTS,
        'census': {
            'pct_20plus':       c['pct_20plus'],
            'high_hrs_carers':  c['high_hrs'],
            'hrs_50_plus':      c['hrs_50_plus'],
            'total_pop_5plus':  c['total'],
            'england_avg_pct':  ENG_20HRS_PCT,
            'source':           'ONS Census 2021 TS039 — LTLA level',
        },
    })

districts.sort(key=lambda x: x['cbi'], reverse=True)

# ── Print quadrant summary ─────────────────────────────────────────────────────
print('\n── Quadrant summary (requires FEP scores from kent-fep-data.json) ──')
print('Load kent-fep-data.json to compute FEP × CBI quadrants.')
print('High CBI districts for reference:')
for d in districts:
    if d['tier'] in ('high','critical'):
        print(f'  {d["name"]:<25} CBI:{d["cbi"]}  {d["tier"]}')

# ── Assemble output JSON ────────────────────────────────────────────────────────
output = {
    'meta': {
        'generated':      datetime.now(timezone.utc).isoformat(),
        'description':    'Kent & Medway Carer Burden Index — Assistiv Systems Layer 2 companion',
        'version':        '1.0',
        'icb':            'NHS Kent and Medway ICB (QKS)',
        'icb_ons_code':   'E54000032',
        'signals': {
            'census_20plus':    {'weight':0.30,'geography':'district','source':'ONS Census 2021 TS039'},
            'social_isolation': {'weight':0.25,'geography':'county',  'source':'NHS Fingertips 90638'},
            'support_access':   {'weight':0.15,'geography':'county',  'source':'NHS Fingertips 90279'},
            'assessment_rate':  {'weight':0.15,'geography':'county',  'source':'NHS Fingertips 93014'},
            'wellbeing':        {'weight':0.15,'geography':'county',  'source':'NHS Fingertips 90283'},
        },
        'methodology': (
            'Signals normalised to 0-100 (50=England average, >50=higher burden). '
            'County-level Fingertips signals applied uniformly across all 13 districts. '
            'District variation driven by ONS Census 2021 20+ hr carer rate (30% weight). '
            'Antidepressant/anxiolytic prescribing excluded — already in FEP model. '
            'DWP Carer Allowance pending — add as Signal 6 when available.'
        ),
        'limitations': (
            'Fingertips carer indicators not available below county level for Kent. '
            'CBI is a population-level triage index, not an individual diagnostic tool. '
            'Census data from 2021 — carer burden has likely increased since then.'
        ),
        'note_on_fep_cbi': (
            'CBI is designed to be read alongside the FEP score, not instead of it. '
            'High FEP + High CBI = frailty risk AND carer system under strain simultaneously. '
            'Use the quadrant map for commissioning decisions.'
        ),
    },
    'county_baseline': fingertips,
    'england_census_avg_20plus_pct': ENG_20HRS_PCT,
    'districts': districts,
}

# ── Commit to GitHub ──────────────────────────────────────────────────────────
def commit_to_github(content, repo, filepath, token):
    api = f'https://api.github.com/repos/{repo}/contents/{filepath}'
    hdrs = {'Authorization': f'token {token}',
            'Accept': 'application/vnd.github.v3+json'}
    b64 = base64.b64encode(json.dumps(content, indent=2).encode()).decode()
    r = requests.get(api, headers=hdrs)
    sha = r.json().get('sha') if r.status_code == 200 else None
    payload = {
        'message': f'CBI v1.0 — {datetime.now(timezone.utc).strftime("%Y-%m-%d")} — 5 signals',
        'content': b64, 'branch': 'main'
    }
    if sha: payload['sha'] = sha
    r = requests.put(api, headers=hdrs, json=payload)
    if r.status_code in (200, 201):
        print(f'Committed ✓')
        print(f'  https://raw.githubusercontent.com/{repo}/main/{filepath}')
        print(f'  {r.json().get("commit",{}).get("html_url","")}')
        return True
    print(f'Commit failed: {r.status_code} — {r.json().get("message","")}')
    return False

print('\nCommitting kent-cbi-data.json to GitHub...')
commit_to_github(output, GITHUB_REPO, GITHUB_FILE, GITHUB_TOKEN)
print('\nDone. Refresh quarterly when new Census or Fingertips data is published.')
print(f'Districts scored: {len(districts)}')
high = sum(1 for d in districts if d["tier"] in ("high","critical"))
print(f'High/critical CBI districts: {high}/13')